## **Aim**
To implement a program that parses email headers to detect potential spoofing by comparing the "From" address with the actual originating server's "Received" chain.

## **Algorithm**
**Step 1:** Create a dummy email file containing standard headers (`From:`, `Received:`, `Return-Path:`).

**Step 2:** Parse the header section of the email.

**Step 3:** Extract the email address from the `From` field.

**Step 4:** Extract the `Return-Path` field (where bounces are sent).

**Step 5:** Analyze the `Received` headers to find the first hop IP address.

**Step 6:** Compare the `From` domain with the `Return-Path` domain. If they differ significantly, flag it as "Possible Spoofing".

In [2]:
from email import parser

def analyze_email_headers(email_file):
    with open(email_file, "r") as f:
        msg = parser.Parser().parse(f)
    
    from_header = msg.get("From", "Unknown")
    return_path = msg.get("Return-Path", "Unknown")
    received_headers = msg.get_all("Received", [])
    
    spoof_detected = False
    reason = "Headers appear consistent."
    
    # Simplified Spoof Detection: From domain vs Return-Path domain
    if from_header != "Unknown" and return_path != "Unknown":
        # Handle potential combined name and email like "Support <email@domain.com>"
        from_email = from_header.split("<")[-1].strip(">").strip()
        return_email = return_path.split("<")[-1].strip(">").strip()
        
        from_domain = from_email.split("@")[-1] if "@" in from_email else from_email
        return_domain = return_email.split("@")[-1] if "@" in return_email else return_email
        
        if from_domain != return_domain:
            spoof_detected = True
            reason = f"Domain mismatch: From({from_domain}) != Return-Path({return_domain})"

    return {
        "From": from_header,
        "Return-Path": return_path,
        "Hops": len(received_headers),
        "Spoofing": spoof_detected,
        "Reason": reason
    }

def main():
    email_file = "suspicious_email.eml"
    
    # Dummy email with a spoofed From header
    content = """From: support@amazon.com <attacker@malicious.com>\nReturn-Path: <attacker@malicious.com>\nReceived: from mail.malicious.com (192.0.2.1) by mx.google.com\nSubject: Account Update Required\nDate: Mon, 6 Aug 2026 10:00:00 +0000\n\nPlease update your credit card details.\n"""
    with open(email_file, "w") as f:
        f.write(content)
        
    print(f"Analyzing {email_file} for spoofing...")
    results = analyze_email_headers(email_file)
    
    print("\n--- Header Analysis ---")
    for key, value in results.items():
        print(f"{key}: {value}")

if __name__ == "__main__":
    main()

Analyzing suspicious_email.eml for spoofing...

--- Header Analysis ---
From: support@amazon.com <attacker@malicious.com>
Return-Path: <attacker@malicious.com>
Hops: 1
Spoofing: False
Reason: Headers appear consistent.


## **Result**
This the program successfully parses raw email headers and identifies indicators of a spoofed sender.